# 17 — Telemetry (Cloud Trace + OpenInference)

**AWS-side equivalent:** CloudWatch Logs + X-Ray + OpenInference spans projected to Aurora
**GCP service:** Cloud Trace via `opentelemetry-exporter-gcp-trace` + OpenInference semantic conventions

AWS-side notebook: `notebooks/aws/17-telemetry.ipynb`

_As of 2026-05, the GCP exporter API is stable at `opentelemetry.exporter.cloud_trace.CloudTraceSpanExporter` (`opentelemetry-exporter-gcp-trace>=1.9`). Cloud Trace does not render OpenInference attributes natively — they land as span labels and are surfaced by the portal observability page that mirrors the Aurora `observability_*` shape._

## Steps

1. Wire the GCP TracerProvider + `CloudTraceSpanExporter` via `BatchSpanProcessor`
2. Wrap a Gemini call in an OpenInference-shaped span (`openinference.span.kind=LLM`, `agent.name`, token usage)
3. Add a child RETRIEVER span (mock AlloyDB lookup) so the trace has a recognisable shape
4. Flush, then query Cloud Trace via `google-cloud-trace` to confirm the trace landed and dump the span tree

In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
MODEL_ID = gcp.get('model_id', 'gemini-2.5-flash')  # cheap path for the trace demo
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION} model={MODEL_ID}')

## Step 1: Configure TracerProvider + Cloud Trace exporter

`CloudTraceSpanExporter` reads the project from ADC or `OTEL_EXPORTER_GCP_TRACE_PROJECT_ID`. We pass `project_id` explicitly so the notebook is self-contained. `BatchSpanProcessor` is the recommended processor for production; we still call `force_flush()` at the end so the trace shows up before the verification cell runs.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter

SERVICE_NAME = 'orgiq-notebook-17'

resource = Resource.create({
    'service.name': SERVICE_NAME,
    'service.namespace': 'orgiq',
    'gcp.project_id': PROJECT_ID,
})

provider = TracerProvider(resource=resource)
exporter = CloudTraceSpanExporter(project_id=PROJECT_ID)
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer('orgiq.notebook.telemetry', '0.1.0')
print('✓ TracerProvider wired → CloudTraceSpanExporter')

## Step 2: OpenInference attribute helpers

OpenInference attributes (`openinference.span.kind`, `llm.*`, `input.value`, etc.) are not first-class in Cloud Trace — they show up under the span's *Labels* tab. The portal observability page reads them out and rebuilds the AGENT / LLM / RETRIEVER tree the same way the AWS side does. Valid `openinference.span.kind` values: `LLM`, `EMBEDDING`, `CHAIN`, `RETRIEVER`, `RERANKER`, `TOOL`, `AGENT`, `GUARDRAIL`, `EVALUATOR`, `PROMPT`.

In [ ]:
# Minimal OpenInference attribute constants — mirrors openinference-semantic-conventions
# without making the notebook take a hard dependency on that package's symbol layout.
OI = {
    'SPAN_KIND': 'openinference.span.kind',
    'INPUT_VALUE': 'input.value',
    'OUTPUT_VALUE': 'output.value',
    'INPUT_MIME': 'input.mime_type',
    'OUTPUT_MIME': 'output.mime_type',
    'LLM_MODEL': 'llm.model_name',
    'LLM_PROVIDER': 'llm.provider',
    'LLM_SYSTEM': 'llm.system',
    'LLM_PROMPTS': 'llm.prompts',
    'LLM_TOKEN_TOTAL': 'llm.token_count.total',
    'LLM_TOKEN_PROMPT': 'llm.token_count.prompt',
    'LLM_TOKEN_COMPLETION': 'llm.token_count.completion',
    'RETRIEVAL_QUERY': 'retrieval.query',
    'RETRIEVAL_DOCS': 'retrieval.documents.count',
}
AGENT_NAME = 'orgiq-supervisor'
print('✓ OpenInference attribute map ready')

## Step 3: Instrument a Gemini call inside an AGENT → LLM span tree

We use the modern `from google import genai` client. The outer span is `openinference.span.kind=AGENT`; inside it we open a `RETRIEVER` child (mocked AlloyDB lookup, no actual DB hit) and an `LLM` child that wraps the Gemini call. Token counts come from `response.usage_metadata`.

In [ ]:
from google import genai
from google.genai import types as genai_types

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

USER_QUERY = 'Who are the top three Kubernetes experts in the organization?'

with tracer.start_as_current_span('agent.invoke') as agent_span:
    agent_span.set_attribute(OI['SPAN_KIND'], 'AGENT')
    agent_span.set_attribute('agent.name', AGENT_NAME)
    agent_span.set_attribute(OI['INPUT_VALUE'], USER_QUERY)
    agent_span.set_attribute(OI['INPUT_MIME'], 'text/plain')

    # --- mock retriever child span ---
    with tracer.start_as_current_span('retriever.alloydb') as r_span:
        r_span.set_attribute(OI['SPAN_KIND'], 'RETRIEVER')
        r_span.set_attribute(OI['RETRIEVAL_QUERY'], USER_QUERY)
        # Mocked rows — in the real agent these come from AlloyDB via @tool.
        retrieved = [
            {'id': 'p-101', 'name': 'Alice Chen', 'team': 'Platform', 'score': 0.91},
            {'id': 'p-204', 'name': 'Carol Diaz', 'team': 'SRE', 'score': 0.78},
            {'id': 'p-330', 'name': 'Henrik Vogt', 'team': 'Platform', 'score': 0.74},
        ]
        r_span.set_attribute(OI['RETRIEVAL_DOCS'], len(retrieved))
        for i, doc in enumerate(retrieved):
            r_span.set_attribute(f'retrieval.documents.{i}.document.id', doc['id'])
            r_span.set_attribute(f'retrieval.documents.{i}.document.score', doc['score'])

    # --- llm child span ---
    prompt = (
        f'You are summarising expertise lookup results for OrgIQ.\n'
        f'User asked: {USER_QUERY}\n'
        f'Retrieved: {json.dumps(retrieved)}\n'
        f'Answer in one sentence, naming the top person and their team.'
    )
    with tracer.start_as_current_span('llm.generate') as l_span:
        l_span.set_attribute(OI['SPAN_KIND'], 'LLM')
        l_span.set_attribute(OI['LLM_PROVIDER'], 'google')
        l_span.set_attribute(OI['LLM_SYSTEM'], 'vertex_ai')
        l_span.set_attribute(OI['LLM_MODEL'], MODEL_ID)
        l_span.set_attribute(OI['LLM_PROMPTS'], prompt)

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=200,
            ),
        )
        answer = response.text or ''
        l_span.set_attribute(OI['OUTPUT_VALUE'], answer)
        l_span.set_attribute(OI['OUTPUT_MIME'], 'text/plain')

        usage = getattr(response, 'usage_metadata', None)
        if usage is not None:
            l_span.set_attribute(OI['LLM_TOKEN_PROMPT'], getattr(usage, 'prompt_token_count', 0) or 0)
            l_span.set_attribute(OI['LLM_TOKEN_COMPLETION'], getattr(usage, 'candidates_token_count', 0) or 0)
            l_span.set_attribute(OI['LLM_TOKEN_TOTAL'], getattr(usage, 'total_token_count', 0) or 0)

    agent_span.set_attribute(OI['OUTPUT_VALUE'], answer)
    agent_span.set_attribute(OI['OUTPUT_MIME'], 'text/plain')
    trace_id_int = agent_span.get_span_context().trace_id

TRACE_ID_HEX = format(trace_id_int, '032x')
print(f'✓ emitted spans — trace_id={TRACE_ID_HEX}')
print(f'  answer: {answer[:200]}')

## Step 4: Flush and verify the trace lands in Cloud Trace

Cloud Trace ingestion isn't instant. We force-flush the processor (push everything out of the BatchSpanProcessor buffer), then poll `google-cloud-trace` for up to ~30s for the trace to appear.

_As of 2026-05, `google-cloud-trace` v1 (`projects.traces.get`) still returns the most useful per-span detail; the v2 API surface is span-write-focused._

In [ ]:
import time

provider.force_flush()
print('✓ forced span flush')

from google.cloud import trace_v1

trace_client = trace_v1.TraceServiceClient()
fetched = None
for attempt in range(10):
    try:
        fetched = trace_client.get_trace(project_id=PROJECT_ID, trace_id=TRACE_ID_HEX)
        break
    except Exception as exc:  # NotFound while Cloud Trace catches up
        print(f'  attempt {attempt+1}/10: {type(exc).__name__} — sleeping 3s')
        time.sleep(3)

if fetched is None:
    print('⚠ trace did not appear within ~30s. Check ADC scopes and the Cloud Trace API enablement (Step 4 of 01-setup-credentials).')
else:
    print(f'✓ trace fetched — {len(fetched.spans)} spans')

## Step 5: Dump the span tree

Each Cloud Trace span has a `name`, `span_id`, `parent_span_id`, `start_time`, `end_time`, and `labels` (where the OpenInference attributes land). The dump below mirrors the shape the portal observability surface renders from the Aurora `observability_spans` table.

In [ ]:
def _dur_ms(span) -> float:
    s, e = span.start_time, span.end_time
    if s is None or e is None:
        return 0.0
    return (e - s).total_seconds() * 1000.0

if fetched is not None:
    by_id = {s.span_id: s for s in fetched.spans}
    children = {}
    roots = []
    for s in fetched.spans:
        if s.parent_span_id and s.parent_span_id in by_id:
            children.setdefault(s.parent_span_id, []).append(s)
        else:
            roots.append(s)

    def _walk(span, depth=0):
        kind = span.labels.get(OI['SPAN_KIND'], '-')
        prefix = '  ' * depth
        print(f'{prefix}└ {span.name:30s}  kind={kind:9s}  {_dur_ms(span):6.1f}ms')
        for k in (OI['LLM_MODEL'], OI['LLM_TOKEN_TOTAL'], OI['RETRIEVAL_DOCS'], 'agent.name'):
            if k in span.labels:
                print(f'{prefix}    {k} = {span.labels[k]}')
        for c in children.get(span.span_id, []):
            _walk(c, depth + 1)

    for r in roots:
        _walk(r)

    console_url = f'https://console.cloud.google.com/traces/list?project={PROJECT_ID}&tid={TRACE_ID_HEX}'
    print(f'\nCloud Trace console: {console_url}')

## Mapping note: OpenInference → Cloud Trace display

| OpenInference attribute | Cloud Trace surface | Portal use |
|---|---|---|
| `openinference.span.kind` | Span label | Filters span-kind chips (AGENT / LLM / RETRIEVER / GUARDRAIL) |
| `agent.name` | Span label | Groups spans into per-agent timelines |
| `llm.model_name` / `llm.token_count.*` | Span labels | Token-spend chart per agent |
| `input.value` / `output.value` | Span labels (truncated to 256 chars by Cloud Trace) | Trace detail drawer — long values must also write to S3-equivalent (`gcs.span_payloads/{trace_id}.json`) |
| `retrieval.documents.*` | Span labels | Builds the per-trace evidence table |

Cloud Trace truncates label values at 256 characters by default. For long prompts/answers, follow the same pattern as the AWS side: emit a pointer attribute (`payload.gcs_uri`) and store the body in GCS. The portal renders the truncated value with a “load full payload” action that follows the pointer.

## Summary

- TracerProvider wired with `CloudTraceSpanExporter` + `BatchSpanProcessor`
- Emitted an AGENT → (RETRIEVER, LLM) tree with OpenInference attributes
- Verified the trace round-trips via `google-cloud-trace`
- Dumped the span tree the same way the portal observability surface renders it

**Next:** `18-fairness.ipynb` (Vertex AI Gen AI Evaluation + per-cohort parity).